# Source-Provenance Reconstruction and Domain-Shift Stress Testing of CXR AI

**Colab-ready experimental notebook for the proposed *Thoracic Radiology* article**

This notebook extends the audit-first workflow supplied with the companion study. It does not reuse the previous internal direct-versus-hierarchical comparison as a new contribution. Its new evidentiary core is:

1. deterministic provenance matching when upstream sources or a manifest are available;
2. uncertainty-labeled latent acquisition domains for unmatched images;
3. quantitative source–label dependence and source-predictability controls;
4. individual source-neutralization ablations;
5. duplicate-group-aware internal and eligible provenance-held-out evaluation;
6. empirical-risk minimization, source-balanced learning, GroupDRO, and CORAL-style domain alignment on frozen image embeddings;
7. calibration, class-conditional conformal prediction, and selective referral under shift;
8. optional locked external validation and a radiologist-review manifest.

All outputs are written to Google Drive in timestamped `Figures/`, `Tables/`, `Models/`, and `Raw/` folders. The notebook never invents an external cohort or forces an invalid three-class held-out analysis when a source lacks class support.


## 1. Experimental guardrails

- Provenance labels are classified as **deterministic**, **manifest**, **latent**, or **unresolved**.
- A latent cluster is never renamed as a hospital, scanner, or repository without documentary evidence.
- Disease labels are not used to choose the number of latent domains.
- A source is eligible for three-class leave-one-source-out testing only when every class reaches the prespecified minimum.
- Calibration and referral thresholds are fitted without held-out-source labels.
- External validation runs only when an independent folder is explicitly supplied.
- The default backbone is frozen and used only for feature extraction. Robust classifiers are trained on cached embeddings.
- Every table is exported as CSV and every figure as PNG. `Outputs_Summary.txt` states what ran, what was skipped, and why.


## 2. Environment, Google Drive, configuration, and dataset retrieval


In [ ]:
import importlib.util, subprocess, sys, os, json, random, hashlib, warnings, gc, shutil, zipfile, time
from pathlib import Path
from datetime import datetime

REQUIRED = {
    "timm": "timm>=1.0.19",
    "imagehash": "ImageHash>=4.3.2",
    "seaborn": "seaborn>=0.13.2",
    "nbformat": "nbformat>=5.10",
}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import imagehash
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps, ImageFilter
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.calibration import calibration_curve
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, balanced_accuracy_score, confusion_matrix,
    f1_score, log_loss, precision_recall_fscore_support, roc_auc_score,
    silhouette_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode, functional as TF

warnings.filterwarnings("ignore", category=UserWarning)

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

STAMP = datetime.now().strftime("%Y_%m_%d_%H_%M")
DRIVE_ROOT = Path("/content/drive/MyDrive") if IN_COLAB else Path.cwd()

CONFIG = {
    "seed": 20260816,
    "dataset_root": "/content/kaggle_data",
    "dataset_slug": "muhammadrehan00/chest-xray-dataset",
    "output_root": str(DRIVE_ROOT / "Outputs" / "Thoracic_Radiology_Source_Aware_CXR" / STAMP),
    "reuse_audit_csv": "",              # Optional path to image_audit.csv from the prior notebook
    "provenance_manifest_csv": "",      # Optional CSV: sha256/path, source_id, assignment_type
    "upstream_roots": {},                # Example: {"Kermany": "/content/drive/MyDrive/Datasets/Kermany"}
    "external_roots": {},                # Example: {"ExternalSite": "/content/drive/MyDrive/Datasets/ExternalCXR"}
    "image_size": 192,
    "batch_size": 128,
    "num_workers": 2,
    "encoder": "mobilenetv3_small_100",
    "near_duplicate_hamming": 4,
    "latent_k_candidates": [2, 3, 4, 5, 6, 7, 8],
    "latent_stability_repeats": 4,
    "latent_sample_max": 6000,
    "minimum_per_class_per_source": 30,
    "minimum_sources_for_lodo": 2,
    "interventions": ["baseline", "aspect_pad", "border_control", "intensity", "compression", "combined"],
    "primary_intervention": "combined",
    "robust_methods": ["ERM", "SourceBalanced", "GroupDRO", "CORAL"],
    "training_seeds": [20260816, 20260817, 20260818],
    "epochs": 45,
    "patience": 8,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "hidden_dim": 128,
    "dropout": 0.15,
    "groupdro_eta": 0.08,
    "coral_lambda": 0.10,
    "bootstrap_replicates": 1000,
    "conformal_alpha": 0.10,
    "selective_entropy_quantile": 0.75,
    "auto_download_results": False,
}

SEED = CONFIG["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

OUTPUT_DIR = Path(CONFIG["output_root"])
FIG_DIR, TAB_DIR, RAW_DIR, MODEL_DIR = [OUTPUT_DIR / name for name in ("Figures", "Tables", "Raw", "Models")]
for directory in (OUTPUT_DIR, FIG_DIR, TAB_DIR, RAW_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "Outputs_Summary.txt"

def log(message=""):
    print(message, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as stream:
        stream.write(str(message) + "\n")

def valid_root(root):
    root = Path(root)
    return all((root / split / label).is_dir() for split in ("train", "val", "test")
               for label in ("normal", "pneumonia", "tuberculosis"))

def locate_root(base):
    base = Path(base)
    candidates = [base]
    if base.exists(): candidates += [p.parent for p in base.rglob("train") if p.is_dir()]
    return next((p.resolve() for p in candidates if valid_root(p)), None)

DATA_ROOT = locate_root(CONFIG["dataset_root"])
if DATA_ROOT is None:
    if importlib.util.find_spec("kagglehub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub>=0.3.12"])
    import kagglehub
    log("Dataset absent; downloading the declared public compilation from Kaggle.")
    downloaded = kagglehub.dataset_download(CONFIG["dataset_slug"], output_dir=CONFIG["dataset_root"])
    DATA_ROOT = locate_root(CONFIG["dataset_root"]) or locate_root(downloaded)
if DATA_ROOT is None:
    raise FileNotFoundError("Expected train/val/test × normal/pneumonia/tuberculosis structure not found.")

(RAW_DIR / "run_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
log(json.dumps({"device": str(DEVICE), "dataset_root": str(DATA_ROOT), "output_root": str(OUTPUT_DIR)}, indent=2))


## 3. Image audit and duplicate-controlled cohort

The audit reproduces the attached notebook's integrity logic and adds technical descriptors used only for provenance characterization. Exact identities with contradictory labels are excluded completely. One representative is retained from a label-consistent exact identity. Conflicting perceptual components are excluded, and every retained identity receives an immutable duplicate-group identifier.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

CLASS_NAMES = ["Normal", "Pneumonia", "Tuberculosis"]
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
FOLDER_TO_CLASS = {name.lower(): name for name in CLASS_NAMES}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def image_entropy(array, bins=64):
    hist, _ = np.histogram(array, bins=bins, range=(0, 1), density=False)
    p = hist / max(hist.sum(), 1); p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def audit_one(record):
    result = dict(record); path = Path(record["path"])
    fields = ["width", "height", "aspect_ratio", "file_kb", "bytes_per_pixel", "mean", "std",
              "p05", "p50", "p95", "sharpness", "dark_fraction", "bright_fraction",
              "border_mean", "center_mean", "center_border_gap", "entropy"]
    result.update({"readable": False, "sha256": "", "pixel_sha256": "", "phash": "",
                   **{f: np.nan for f in fields}, "error": ""})
    try:
        raw = path.read_bytes()
        with Image.open(path) as image:
            image.load(); gray = image.convert("L"); width, height = gray.size
            thumb = gray.resize((192, 192), Image.Resampling.BILINEAR)
            arr_u8 = np.asarray(thumb, dtype=np.uint8)
            arr = arr_u8.astype(np.float32) / 255.0
            lap = -4*arr + np.roll(arr,1,0)+np.roll(arr,-1,0)+np.roll(arr,1,1)+np.roll(arr,-1,1)
            b = 12
            border = np.concatenate([arr[:b].ravel(), arr[-b:].ravel(), arr[:, :b].ravel(), arr[:, -b:].ravel()])
            center = arr[48:144, 48:144]
            result.update({
                "readable": True,
                "sha256": hashlib.sha256(raw).hexdigest(),
                "pixel_sha256": hashlib.sha256(np.asarray(gray).tobytes()).hexdigest(),
                "phash": str(imagehash.phash(gray)),
                "width": width, "height": height, "aspect_ratio": width/max(height,1),
                "file_kb": len(raw)/1024, "bytes_per_pixel": len(raw)/max(width*height,1),
                "mean": float(arr.mean()), "std": float(arr.std()),
                "p05": float(np.quantile(arr,.05)), "p50": float(np.quantile(arr,.50)), "p95": float(np.quantile(arr,.95)),
                "sharpness": float(lap.var()), "dark_fraction": float((arr < .02).mean()),
                "bright_fraction": float((arr > .98).mean()), "border_mean": float(border.mean()),
                "center_mean": float(center.mean()), "center_border_gap": float(center.mean()-border.mean()),
                "entropy": image_entropy(arr),
            })
    except Exception as exc:
        result["error"] = repr(exc)
    return result

records = []
for supplied_split in ("train", "val", "test"):
    for folder, class_name in FOLDER_TO_CLASS.items():
        for path in sorted((DATA_ROOT / supplied_split / folder).glob("*")):
            if path.suffix.lower() in IMAGE_SUFFIXES:
                records.append({"path": str(path), "supplied_split": supplied_split, "class_name": class_name})
inventory = pd.DataFrame(records)

reuse = Path(CONFIG["reuse_audit_csv"]) if CONFIG["reuse_audit_csv"] else None
audit_file = TAB_DIR / "image_audit.csv"
technical_audit_fields={"width","height","aspect_ratio","file_kb","bytes_per_pixel","mean","std","p05","p50","p95",
                        "sharpness","dark_fraction","bright_fraction","border_mean","center_mean","center_border_gap","entropy"}
required_audit_columns=set(inventory.columns)|technical_audit_fields|{"sha256","pixel_sha256","phash","readable"}
if reuse and reuse.exists():
    audit_df = pd.read_csv(reuse)
    missing_cols = required_audit_columns - set(audit_df.columns)
    if missing_cols:
        log(f"Prior audit lacks new provenance descriptors {sorted(missing_cols)}; recomputing the enriched audit.")
        audit_df = pd.DataFrame()
    else:
        log(f"Reused audit table: {reuse}")
else:
    audit_df = pd.DataFrame()
if audit_df.empty:
    completed = []
    with ThreadPoolExecutor(max_workers=4) as pool:
        for i, result in enumerate(pool.map(audit_one, inventory.to_dict("records")), 1):
            completed.append(result)
            if i % 500 == 0 or i == len(inventory): log(f"Audited {i:,}/{len(inventory):,}")
    audit_df = pd.DataFrame(completed)
    audit_df.to_csv(audit_file, index=False)

audit_df["readable"] = audit_df["readable"].astype(bool)
valid = audit_df.loc[audit_df.readable].copy().reset_index(drop=True)
exact_summary = valid.groupby("sha256").agg(n=("path","size"), n_classes=("class_name","nunique")).reset_index()
conflict_hashes = set(exact_summary.loc[exact_summary.n_classes > 1, "sha256"])
exact_conflicts = valid.loc[valid.sha256.isin(conflict_hashes)].copy()
exact_conflicts.to_csv(TAB_DIR / "excluded_exact_label_conflicts.csv", index=False)
unique_df = (valid.loc[~valid.sha256.isin(conflict_hashes)].sort_values(["sha256","path"])
             .drop_duplicates("sha256").reset_index(drop=True))

class UnionFind:
    def __init__(self, n): self.parent=list(range(n)); self.rank=[0]*n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def union(self, a, b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.rank[a]<self.rank[b]:a,b=b,a
        self.parent[b]=a
        if self.rank[a]==self.rank[b]:self.rank[a]+=1

class BKNode:
    def __init__(self,value,index): self.value=value; self.indices=[index]; self.children={}
class BKTree:
    def __init__(self): self.root=None
    def add(self,value,index):
        if self.root is None:self.root=BKNode(value,index);return
        node=self.root
        while True:
            d=value-node.value
            if d==0:node.indices.append(index);return
            if d not in node.children:node.children[d]=BKNode(value,index);return
            node=node.children[d]
    def query(self,value,radius):
        if self.root is None:return []
        found=[]; stack=[self.root]
        while stack:
            node=stack.pop(); d=value-node.value
            if d<=radius:found.extend(node.indices)
            stack.extend(child for edge,child in node.children.items() if d-radius<=edge<=d+radius)
        return found

uf=UnionFind(len(unique_df)); tree=BKTree()
for index,text_hash in enumerate(unique_df.phash):
    value=imagehash.hex_to_hash(text_hash)
    for neighbor in tree.query(value,CONFIG["near_duplicate_hamming"]):uf.union(index,neighbor)
    tree.add(value,index)
roots=[uf.find(i) for i in range(len(unique_df))]
mapping={root:n for n,root in enumerate(sorted(set(roots)))}
unique_df["duplicate_group"]=[mapping[root] for root in roots]
component_summary=unique_df.groupby("duplicate_group").agg(n=("path","size"),n_classes=("class_name","nunique")).reset_index()
near_conflict_groups=set(component_summary.loc[component_summary.n_classes>1,"duplicate_group"])
near_conflicts=unique_df.loc[unique_df.duplicate_group.isin(near_conflict_groups)].copy()
near_conflicts.to_csv(TAB_DIR / "excluded_near_duplicate_label_conflicts.csv", index=False)
clean_df=unique_df.loc[~unique_df.duplicate_group.isin(near_conflict_groups)].copy().reset_index(drop=True)
clean_df["label"]=clean_df.class_name.map(CLASS_TO_INDEX).astype(int)
clean_df["row_id"]=np.arange(len(clean_df))

integrity={"source_files":len(audit_df),"readable_files":len(valid),"unique_sha256":int(valid.sha256.nunique()),
           "exact_conflict_identities":len(conflict_hashes),"exact_conflict_files":len(exact_conflicts),
           "near_conflict_groups":len(near_conflict_groups),"near_conflict_images":len(near_conflicts),
           "retained_images":len(clean_df),"class_counts":clean_df.class_name.value_counts().to_dict()}
(RAW_DIR/"integrity_audit.json").write_text(json.dumps(integrity,indent=2),encoding="utf-8")
log("Integrity audit:\n"+json.dumps(integrity,indent=2))


## 4. Deterministic provenance matching and latent-domain inference

Three routes are applied in order:

1. an optional author-verified manifest;
2. exact decoded-pixel or byte matching against optional upstream roots;
3. latent acquisition-domain clustering for still-unmatched images.

The clustering uses technical descriptors only. Candidate domain counts are evaluated using silhouette and stability. Disease labels are inspected only after the domain solution has been frozen.


In [ ]:
TECH_COLS = ["width","height","aspect_ratio","file_kb","bytes_per_pixel","mean","std","p05","p50","p95",
             "sharpness","dark_fraction","bright_fraction","border_mean","center_mean","center_border_gap","entropy"]

clean_df["source_id"] = pd.NA
clean_df["assignment_type"] = pd.NA
clean_df["assignment_confidence"] = np.nan

# Route 1: verified manifest.
manifest_path = Path(CONFIG["provenance_manifest_csv"]) if CONFIG["provenance_manifest_csv"] else None
if manifest_path and manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    key = "sha256" if "sha256" in manifest.columns else "path"
    required = {key,"source_id"}
    if not required.issubset(manifest): raise ValueError(f"Manifest must contain {required}")
    manifest = manifest.drop_duplicates(key)
    clean_df = clean_df.merge(manifest[[c for c in (key,"source_id","assignment_type") if c in manifest]],
                              on=key,how="left",suffixes=("","_manifest"))
    clean_df["source_id"] = clean_df["source_id_manifest"].combine_first(clean_df["source_id"])
    if "assignment_type_manifest" in clean_df:
        clean_df["assignment_type"] = clean_df["assignment_type_manifest"].fillna("manifest")
    clean_df.loc[clean_df.source_id.notna() & clean_df.assignment_type.isna(),"assignment_type"]="manifest"
    clean_df.loc[clean_df.source_id.notna(),"assignment_confidence"]=1.0
    clean_df.drop(columns=[c for c in clean_df if c.endswith("_manifest")],inplace=True)

# Route 2: exact matching to optional upstream roots.
def scan_upstream_hashes(source_name, root):
    rows=[]
    files=[p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]
    for i,path in enumerate(files,1):
        try:
            raw=path.read_bytes()
            with Image.open(path) as im:
                pixel_hash=hashlib.sha256(np.asarray(im.convert("L")).tobytes()).hexdigest()
            rows.append({"source_id":source_name,"upstream_path":str(path),"sha256":hashlib.sha256(raw).hexdigest(),"pixel_sha256":pixel_hash})
        except Exception: pass
        if i%1000==0: log(f"Indexed {i:,}/{len(files):,} upstream files for {source_name}")
    return pd.DataFrame(rows)

upstream_tables=[]
for source_name,root in CONFIG["upstream_roots"].items():
    if Path(root).exists(): upstream_tables.append(scan_upstream_hashes(source_name,root))
if upstream_tables:
    upstream=pd.concat(upstream_tables,ignore_index=True)
    byte_unique=upstream.groupby("sha256").filter(lambda d:d.source_id.nunique()==1).drop_duplicates("sha256")
    pixel_unique=upstream.groupby("pixel_sha256").filter(lambda d:d.source_id.nunique()==1).drop_duplicates("pixel_sha256")
    byte_map=byte_unique.set_index("sha256").source_id
    pixel_map=pixel_unique.set_index("pixel_sha256").source_id
    unmatched=clean_df.source_id.isna()
    clean_df.loc[unmatched,"source_id"]=clean_df.loc[unmatched,"sha256"].map(byte_map)
    exact_now=unmatched & clean_df.source_id.notna()
    clean_df.loc[exact_now,"assignment_type"]="deterministic_byte"
    unmatched=clean_df.source_id.isna()
    clean_df.loc[unmatched,"source_id"]=clean_df.loc[unmatched,"pixel_sha256"].map(pixel_map)
    pixel_now=unmatched & clean_df.source_id.notna()
    clean_df.loc[pixel_now,"assignment_type"]="deterministic_pixel"
    clean_df.loc[exact_now|pixel_now,"assignment_confidence"]=1.0
    upstream.to_csv(TAB_DIR/"upstream_hash_index.csv",index=False)

# Route 3: stable latent domains for unmatched images.
unmatched_idx=clean_df.index[clean_df.source_id.isna()].to_numpy()
latent_selection=[]
if len(unmatched_idx):
    tech=clean_df.loc[unmatched_idx,TECH_COLS].replace([np.inf,-np.inf],np.nan)
    tech=tech.fillna(tech.median())
    scaler=StandardScaler(); z=scaler.fit_transform(tech)
    n_components=min(10,z.shape[1],max(2,z.shape[0]-1))
    pca=PCA(n_components=n_components,random_state=SEED); z_pca=pca.fit_transform(z)
    rng=np.random.default_rng(SEED)
    eval_idx=rng.choice(len(z_pca),min(CONFIG["latent_sample_max"],len(z_pca)),replace=False)
    best=None
    for k in CONFIG["latent_k_candidates"]:
        if k>=len(eval_idx):continue
        labels=[]
        for repeat in range(CONFIG["latent_stability_repeats"]):
            labels.append(KMeans(n_clusters=k,n_init=20,random_state=SEED+repeat).fit_predict(z_pca[eval_idx]))
        stability=np.mean([adjusted_rand_score(labels[0],x) for x in labels[1:]]) if len(labels)>1 else 1.0
        silhouette=silhouette_score(z_pca[eval_idx],labels[0],sample_size=min(3000,len(eval_idx)),random_state=SEED)
        score=0.6*silhouette+0.4*stability
        row={"k":k,"silhouette":silhouette,"stability_ari":stability,"selection_score":score}
        latent_selection.append(row)
        if best is None or score>best[0]:best=(score,k)
    if best is None: raise RuntimeError("No feasible latent-domain count.")
    chosen_k=best[1]
    km=KMeans(n_clusters=chosen_k,n_init=40,random_state=SEED).fit(z_pca)
    distances=km.transform(z_pca); order=np.sort(distances,axis=1)
    confidence=1-order[:,0]/np.maximum(order[:,1],1e-8)
    clean_df.loc[unmatched_idx,"source_id"]=[f"latent_L{x+1}" for x in km.labels_]
    clean_df.loc[unmatched_idx,"assignment_type"]="latent"
    clean_df.loc[unmatched_idx,"assignment_confidence"]=confidence
    joblib.dump({"scaler":scaler,"pca":pca,"kmeans":km,"technical_columns":TECH_COLS},MODEL_DIR/"latent_domain_model.joblib")

pd.DataFrame(latent_selection).to_csv(TAB_DIR/"latent_domain_selection.csv",index=False)
clean_df["source_id"]=clean_df.source_id.fillna("unresolved").astype(str)
clean_df["assignment_type"]=clean_df.assignment_type.fillna("unresolved").astype(str)
clean_df.to_csv(TAB_DIR/"provenance_manifest_full.csv",index=False)

source_class=(clean_df.groupby(["source_id","class_name"]).size().unstack(fill_value=0).reindex(columns=CLASS_NAMES,fill_value=0))
source_class["Total"]=source_class.sum(axis=1)
source_class.to_csv(TAB_DIR/"source_by_class_counts.csv")
assignment_summary=(clean_df.groupby(["source_id","assignment_type"]).agg(
    n_images=("path","size"),assignment_confidence_median=("assignment_confidence","median"),
    width_median=("width","median"),height_median=("height","median")).reset_index())
assignment_summary=assignment_summary.merge(source_class[CLASS_NAMES].reset_index(),on="source_id",how="left")
assignment_summary.to_csv(TAB_DIR/"Table1_Provenance_Characteristics.csv",index=False)

mapping_review=(clean_df.groupby("source_id",group_keys=False).apply(
    lambda d:d.sample(min(20,len(d)),random_state=SEED)).reset_index(drop=True))
mapping_review=mapping_review[["row_id","path","sha256","source_id","assignment_type","assignment_confidence","class_name"]]
for col in ["technical_match_plausible","orientation_projection_notes","border_annotation_notes","reviewer_comment"]:mapping_review[col]=""
mapping_review.to_csv(TAB_DIR/"provenance_mapping_review_manifest.csv",index=False)
log("Source × class support:\n"+source_class.to_string())


## 5. Source–label association and eligibility gate

This cell quantifies source–class dependence and decides whether the primary three-class provenance-held-out experiment is estimable. If fewer than two sources contain the minimum number of examples from every class, the notebook records a **go/no-go failure** and does not manufacture three-class source-held-out scores.


In [ ]:
from scipy.stats import chi2_contingency
from sklearn.metrics import normalized_mutual_info_score

def cramers_v(table):
    chi2=chi2_contingency(table,correction=False)[0]; n=table.sum(); r,k=table.shape
    phi2=chi2/max(n,1); correction=((k-1)*(r-1))/max(n-1,1)
    phi2corr=max(0,phi2-correction); rcorr=r-((r-1)**2)/max(n-1,1); kcorr=k-((k-1)**2)/max(n-1,1)
    return float(np.sqrt(phi2corr/max(min(kcorr-1,rcorr-1),1e-12)))

source_codes=pd.Categorical(clean_df.source_id).codes
association={
    "cramers_v":cramers_v(pd.crosstab(clean_df.source_id,clean_df.class_name).to_numpy()),
    "normalized_mutual_information":normalized_mutual_info_score(source_codes,clean_df.label),
    "n_sources":int(clean_df.source_id.nunique()),
}
pd.DataFrame([association]).to_csv(TAB_DIR/"source_label_association.csv",index=False)

minimum=CONFIG["minimum_per_class_per_source"]
eligible_sources=[s for s in source_class.index[(source_class[CLASS_NAMES]>=minimum).all(axis=1)].tolist() if s!="unresolved"]
eligibility=(source_class.reset_index().assign(
    eligible_three_class=lambda d:d["source_id"].isin(eligible_sources),minimum_required=minimum))
eligibility.to_csv(TAB_DIR/"source_heldout_eligibility.csv",index=False)
LODO_VALID=len(eligible_sources)>=CONFIG["minimum_sources_for_lodo"]
guardrail={"eligible_sources":eligible_sources,"n_eligible":len(eligible_sources),"three_class_lodo_valid":LODO_VALID,
           "reason":"adequate class support" if LODO_VALID else "insufficient multi-source support across all three classes"}
(RAW_DIR/"source_heldout_guardrail.json").write_text(json.dumps(guardrail,indent=2),encoding="utf-8")
log("Source–label association: "+json.dumps(association))
log("Three-class source-held-out gate: "+json.dumps(guardrail))


## 6. Duplicate-group-aware internal partitions

The supplied train/validation/test folders are not trusted as experimental partitions. New training, validation, calibration, and locked internal-test subsets are reconstructed while keeping every duplicate group intact.


In [ ]:
def grouped_holdout(frame,n_splits,seed):
    splitter=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    keep,take=next(splitter.split(frame,frame.label,frame.duplicate_group))
    return frame.iloc[keep].copy().reset_index(drop=True),frame.iloc[take].copy().reset_index(drop=True)

remaining,internal_test_df=grouped_holdout(clean_df,10,SEED)
remaining,internal_cal_df=grouped_holdout(remaining,9,SEED+1)
internal_train_df,internal_val_df=grouped_holdout(remaining,8,SEED+2)
internal_frames={"train":internal_train_df,"validation":internal_val_df,"calibration":internal_cal_df,"test":internal_test_df}
internal_assignment=pd.concat([f.assign(experimental_split=k) for k,f in internal_frames.items()],ignore_index=True)
assert internal_assignment.groupby("sha256").experimental_split.nunique().max()==1
assert internal_assignment.groupby("duplicate_group").experimental_split.nunique().max()==1
internal_assignment.to_csv(TAB_DIR/"internal_split_assignment.csv",index=False)
internal_assignment.groupby(["experimental_split","class_name","source_id"]).size().rename("n").reset_index().to_csv(
    TAB_DIR/"internal_split_counts.csv",index=False)


## 7. Source-neutralization interventions and cached frozen embeddings

The interventions are evaluated individually. `combined` applies aspect-preserving resize, technical border control, intensity standardization, and a fixed JPEG round trip. Lung-field cropping is deliberately absent unless a validated segmentation model is supplied; it must not be simulated by a crude center crop.


In [ ]:
IMAGENET_MEAN=[0.485,0.456,0.406]; IMAGENET_STD=[0.229,0.224,0.225]

def border_control(image, tolerance=7):
    gray=np.asarray(image.convert("L"),dtype=np.uint8)
    h,w=gray.shape; edge=np.concatenate([gray[0],gray[-1],gray[:,0],gray[:,-1]])
    background=float(np.median(edge)); mask=np.abs(gray.astype(float)-background)>tolerance
    ys,xs=np.where(mask)
    if len(xs)<0.25*w*h:return image
    x0,x1=max(0,xs.min()-2),min(w,xs.max()+3); y0,y1=max(0,ys.min()-2),min(h,ys.max()+3)
    return image.crop((x0,y0,x1,y1))

def jpeg_equalize(image, quality=90):
    import io
    buffer=io.BytesIO(); image.save(buffer,format="JPEG",quality=quality,optimize=False,subsampling=0);buffer.seek(0)
    out=Image.open(buffer).convert("RGB");out.load();return out

def aspect_pad(image,size):
    image=ImageOps.contain(image,(size,size),method=Image.Resampling.BILINEAR)
    canvas=Image.new("RGB",(size,size),(0,0,0));canvas.paste(image,((size-image.width)//2,(size-image.height)//2));return canvas

class SourceTransform:
    def __init__(self,name,size):self.name=name;self.size=size
    def __call__(self,image):
        name=self.name.split("_external_")[0]
        image=image.convert("RGB")
        if name in ("border_control","combined"):image=border_control(image)
        if name in ("intensity","combined"):image=ImageOps.autocontrast(image.convert("L"),cutoff=1).convert("RGB")
        if name in ("compression","combined"):image=jpeg_equalize(image)
        if name=="baseline":image=image.resize((self.size,self.size),Image.Resampling.BILINEAR)
        else:image=aspect_pad(image,self.size)
        tensor=TF.to_tensor(image);return TF.normalize(tensor,IMAGENET_MEAN,IMAGENET_STD)

class CXRDataset(Dataset):
    def __init__(self,frame,transform):self.paths=frame.path.tolist();self.labels=frame.label.to_numpy(np.int64);self.transform=transform
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as image:tensor=self.transform(image)
        return tensor,int(self.labels[index]),index

def extract_embeddings(frame,intervention,encoder):
    cache_file=RAW_DIR/f"embeddings_{intervention}.npz"
    if cache_file.exists():
        cached=np.load(cache_file);return cached["features"],cached["labels"]
    dataset=CXRDataset(frame,SourceTransform(intervention,CONFIG["image_size"]))
    loader=DataLoader(dataset,batch_size=CONFIG["batch_size"],shuffle=False,num_workers=CONFIG["num_workers"],
                      pin_memory=DEVICE.type=="cuda",persistent_workers=CONFIG["num_workers"]>0)
    feature_batches=[];label_batches=[]
    with torch.inference_mode():
        for batch,(images,labels,_) in enumerate(loader,1):
            with torch.amp.autocast(device_type=DEVICE.type,enabled=AMP_ENABLED):features=encoder(images.to(DEVICE,non_blocking=True))
            feature_batches.append(features.float().cpu().numpy());label_batches.append(labels.numpy())
            if batch%20==0:log(f"{intervention}: {min(batch*CONFIG['batch_size'],len(frame)):,}/{len(frame):,}")
    features=np.concatenate(feature_batches);labels=np.concatenate(label_batches)
    np.savez_compressed(cache_file,features=features,labels=labels)
    return features,labels

encoder=timm.create_model(CONFIG["encoder"],pretrained=True,num_classes=0,global_pool="avg").to(DEVICE).eval()
for parameter in encoder.parameters():parameter.requires_grad=False
all_embeddings={}
for intervention in CONFIG["interventions"]:
    all_embeddings[intervention],labels_check=extract_embeddings(clean_df,intervention,encoder)
    assert np.array_equal(labels_check,clean_df.label.to_numpy())
encoder.cpu();del encoder;gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

PRIMARY_X=all_embeddings[CONFIG["primary_intervention"]]


## 8. Source predictability and intervention ablation

Each intervention is judged on two axes: disease macro-F1 and residual source predictability. A mitigation is not declared successful merely because it reduces source prediction; disease performance and source-held-out behavior must be preserved.


In [ ]:
def rows(frame):return frame.row_id.to_numpy(dtype=int)
def safe_macro_auc(y,p):
    try:return roc_auc_score(y,p,multi_class="ovr",average="macro")
    except Exception:return np.nan
def basic_metrics(y,p):
    pred=p.argmax(1)
    return {"accuracy":accuracy_score(y,pred),"balanced_accuracy":balanced_accuracy_score(y,pred),
            "macro_f1":f1_score(y,pred,average="macro"),"macro_auc_ovr":safe_macro_auc(y,p)}

source_names=sorted(clean_df.source_id.unique())
source_to_index={s:i for i,s in enumerate(source_names)}
source_y=clean_df.source_id.map(source_to_index).to_numpy()
train_idx, test_idx=rows(internal_train_df), rows(internal_test_df)

source_predictability=[]; intervention_rows=[]
metadata_model=make_pipeline(StandardScaler(),LogisticRegression(class_weight="balanced",max_iter=500))
metadata_model.fit(clean_df.loc[train_idx,TECH_COLS],source_y[train_idx])
metadata_pred=metadata_model.predict(clean_df.loc[test_idx,TECH_COLS])
source_predictability.append({"representation":"native_metadata","source_macro_f1":f1_score(source_y[test_idx],metadata_pred,average="macro")})

for name,X in all_embeddings.items():
    disease=make_pipeline(StandardScaler(),LogisticRegression(class_weight="balanced",max_iter=500))
    disease.fit(X[train_idx],clean_df.label.to_numpy()[train_idx]);p=disease.predict_proba(X[test_idx])
    disease_result=basic_metrics(clean_df.label.to_numpy()[test_idx],p)
    source_model=make_pipeline(StandardScaler(),LogisticRegression(class_weight="balanced",max_iter=500))
    source_model.fit(X[train_idx],source_y[train_idx]);source_pred=source_model.predict(X[test_idx])
    source_f1=f1_score(source_y[test_idx],source_pred,average="macro")
    source_predictability.append({"representation":name,"source_macro_f1":source_f1})
    intervention_rows.append({"intervention":name,**disease_result,"source_macro_f1":source_f1})

source_predictability_df=pd.DataFrame(source_predictability)
intervention_df=pd.DataFrame(intervention_rows)
source_predictability_df.to_csv(TAB_DIR/"source_predictability.csv",index=False)
intervention_df.to_csv(TAB_DIR/"source_neutralization_ablation.csv",index=False)
log("Intervention ablation:\n"+intervention_df.round(5).to_string(index=False))


## 9. Matched-capacity robust classifiers

All four strategies use the same small neural classifier on frozen embeddings:

- **ERM:** class-weighted empirical risk minimization;
- **SourceBalanced:** inverse source–class weighting;
- **GroupDRO:** dynamically upweights high-loss acquisition groups;
- **CORAL:** aligns hidden-representation means and covariances across observed training sources.

Model selection uses validation macro-F1 and worst-source macro-F1. No held-out-source label affects selection.


In [ ]:
class RobustMLP(nn.Module):
    def __init__(self,d_in,hidden,n_classes):
        super().__init__();self.feature=nn.Sequential(nn.Linear(d_in,hidden),nn.ReLU(),nn.Dropout(CONFIG["dropout"]));self.head=nn.Linear(hidden,n_classes)
    def forward(self,x):
        h=self.feature(x);return self.head(h),h

def worst_source_f1(y,pred,domains):
    values=[]
    for d in np.unique(domains):
        m=domains==d
        if set(np.unique(y[m]))==set(range(len(CLASS_NAMES))):
            values.append(f1_score(y[m],pred[m],average="macro",labels=np.arange(len(CLASS_NAMES)),zero_division=0))
    return float(min(values)) if values else np.nan

def coral_penalty(h,domains):
    stats=[]
    for d in torch.unique(domains):
        hd=h[domains==d]
        if len(hd)<3:continue
        mean=hd.mean(0);centered=hd-mean;cov=centered.T@centered/(len(hd)-1);stats.append((mean,cov))
    if len(stats)<2:return h.new_tensor(0.)
    losses=[]
    for i in range(len(stats)):
        for j in range(i+1,len(stats)):
            losses.append((stats[i][0]-stats[j][0]).pow(2).mean()+(stats[i][1]-stats[j][1]).pow(2).mean())
    return torch.stack(losses).mean()

def domain_class_weights(y,domains):
    keys=pd.Series([f"{d}|{c}" for d,c in zip(domains,y)]);counts=keys.value_counts();w=keys.map(lambda x:1/counts[x]).to_numpy(float)
    return w/w.mean()

def train_robust(method,X_train,y_train,d_train,X_val,y_val,d_val,seed):
    torch.manual_seed(seed);np.random.seed(seed)
    scaler=StandardScaler();xt=scaler.fit_transform(X_train).astype("float32");xv=scaler.transform(X_val).astype("float32")
    model=RobustMLP(xt.shape[1],CONFIG["hidden_dim"],len(CLASS_NAMES)).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=CONFIG["learning_rate"],weight_decay=CONFIG["weight_decay"])
    class_counts=np.bincount(y_train,minlength=len(CLASS_NAMES));class_w=len(y_train)/(len(CLASS_NAMES)*np.maximum(class_counts,1))
    class_w=torch.tensor(class_w,dtype=torch.float32,device=DEVICE)
    sample_w=domain_class_weights(y_train,d_train) if method=="SourceBalanced" else np.ones(len(y_train))
    domain_values=np.unique(d_train);domain_map={d:i for i,d in enumerate(domain_values)}
    q=torch.ones(len(domain_values),device=DEVICE)/max(len(domain_values),1)
    rng=np.random.default_rng(seed);best_state=None;best_score=-np.inf;wait=0
    batch_size=min(512,len(xt))
    for epoch in range(CONFIG["epochs"]):
        model.train();order=rng.permutation(len(xt))
        for start in range(0,len(order),batch_size):
            ix=order[start:start+batch_size]
            xb=torch.from_numpy(xt[ix]).to(DEVICE);yb=torch.from_numpy(y_train[ix]).long().to(DEVICE)
            db=torch.tensor([domain_map[d] for d in d_train[ix]],dtype=torch.long,device=DEVICE)
            logits,h=model(xb);per=F.cross_entropy(logits,yb,weight=class_w,reduction="none")
            if method=="SourceBalanced":loss=(per*torch.tensor(sample_w[ix],dtype=torch.float32,device=DEVICE)).mean()
            elif method=="GroupDRO":
                gl=[]
                for g in range(len(domain_values)):
                    m=db==g;gl.append(per[m].mean() if m.any() else per.new_tensor(0.))
                gl=torch.stack(gl);q=q*torch.exp(CONFIG["groupdro_eta"]*gl.detach());q=q/q.sum();loss=(q*gl).sum()
            elif method=="CORAL":loss=per.mean()+CONFIG["coral_lambda"]*coral_penalty(h,db)
            else:loss=per.mean()
            optimizer.zero_grad();loss.backward();optimizer.step()
        model.eval()
        with torch.inference_mode():pv=F.softmax(model(torch.from_numpy(xv).to(DEVICE))[0],1).cpu().numpy()
        pred=pv.argmax(1);macro=f1_score(y_val,pred,average="macro");worst=worst_source_f1(y_val,pred,d_val)
        score=.6*macro+.4*(worst if np.isfinite(worst) else macro)
        if score>best_score+1e-5:
            best_score=score;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};wait=0
        else:wait+=1
        if wait>=CONFIG["patience"]:break
    model.load_state_dict(best_state);return {"model":model.cpu(),"scaler":scaler,"selection_score":best_score,"method":method,"seed":seed}

def predict_bundle(bundle,X):
    model=bundle["model"].to(DEVICE).eval();x=bundle["scaler"].transform(X).astype("float32")
    with torch.inference_mode():p=F.softmax(model(torch.from_numpy(x).to(DEVICE))[0],1).cpu().numpy()
    model.cpu();return p

def split_training_pool(frame,seed):
    remaining,cal=grouped_holdout(frame,8,seed)
    train,val=grouped_holdout(remaining,7,seed+1)
    return train,val,cal


## 10. Internal and provenance-held-out experiments

The internal protocol is retained as a comparator. Eligible provenance groups are then held out one at a time. Results are saved per seed, method, protocol, and held-out source. If the eligibility gate failed, only the internal comparator is run and the limitation is written to the summary.


In [ ]:
X=PRIMARY_X;y=clean_df.label.to_numpy();domains=clean_df.source_id.map(source_to_index).to_numpy()
experiment_rows=[];prediction_rows=[];trained_bundles={}

def run_experiment(protocol,held_source,train_frame,val_frame,cal_frame,test_frame):
    for method in CONFIG["robust_methods"]:
        for seed in CONFIG["training_seeds"]:
            bundle=train_robust(method,X[rows(train_frame)],y[rows(train_frame)],domains[rows(train_frame)],
                                X[rows(val_frame)],y[rows(val_frame)],domains[rows(val_frame)],seed)
            p=predict_bundle(bundle,X[rows(test_frame)]);m=basic_metrics(y[rows(test_frame)],p)
            pred=p.argmax(1);m["worst_source_macro_f1"]=worst_source_f1(y[rows(test_frame)],pred,domains[rows(test_frame)])
            experiment_rows.append({"protocol":protocol,"held_source":held_source,"method":method,"seed":seed,
                                    "n_test":len(test_frame),**m})
            for local_i,row_id in enumerate(rows(test_frame)):
                prediction_rows.append({"protocol":protocol,"held_source":held_source,"method":method,"seed":seed,
                    "row_id":int(row_id),"true_label":int(y[row_id]),"source_id":clean_df.loc[row_id,"source_id"],
                    **{f"p_{CLASS_NAMES[k].lower()}":float(p[local_i,k]) for k in range(3)}})
            key=(protocol,held_source,method,seed);trained_bundles[key]=(bundle,cal_frame)
            log(f"{protocol} | {held_source} | {method} | seed {seed}: macro-F1={m['macro_f1']:.4f}")

run_experiment("internal","internal_test",internal_train_df,internal_val_df,internal_cal_df,internal_test_df)

if LODO_VALID:
    for source in eligible_sources:
        test_frame=clean_df.loc[clean_df.source_id==source].copy().reset_index(drop=True)
        pool=clean_df.loc[clean_df.source_id!=source].copy().reset_index(drop=True)
        if pool.class_name.nunique()<3:
            log(f"Skipped {source}: training pool lacks all classes.");continue
        train_frame,val_frame,cal_frame=split_training_pool(pool,SEED+source_to_index[source]*17)
        run_experiment("provenance_held_out",source,train_frame,val_frame,cal_frame,test_frame)
else:
    log("Three-class provenance-held-out experiment skipped by the prespecified class-support gate.")

experiments=pd.DataFrame(experiment_rows);predictions=pd.DataFrame(prediction_rows)
experiments.to_csv(TAB_DIR/"seed_level_discrimination.csv",index=False)
predictions.to_csv(RAW_DIR/"case_level_predictions.csv",index=False)

discrimination_summary=(experiments.groupby(["protocol","held_source","method"]).agg(
    n_test=("n_test","max"),macro_f1_mean=("macro_f1","mean"),macro_f1_sd=("macro_f1","std"),
    balanced_accuracy_mean=("balanced_accuracy","mean"),macro_auc_mean=("macro_auc_ovr","mean"),
    worst_source_macro_f1_mean=("worst_source_macro_f1","mean")).reset_index())
discrimination_summary.to_csv(TAB_DIR/"Table2_Discrimination_Summary.csv",index=False)


## 11. Locked calibration, conformal prediction, and selective referral

Temperature, class-conditional conformal thresholds, and the entropy referral threshold are fitted on the calibration subset associated with each trained model. They are then applied unchanged to the corresponding internal or held-out-source test set.


In [ ]:
def logits_from_probs(p):return np.log(np.clip(p,1e-8,1))
def fit_temperature(p,y_true):
    logits=logits_from_probs(p)
    result=minimize_scalar(lambda log_t:log_loss(y_true,softmax(logits/np.exp(log_t),axis=1),labels=np.arange(3)),
                           bounds=(-3,3),method="bounded")
    return float(np.exp(result.x))
def calibrate(p,t):return softmax(logits_from_probs(p)/t,axis=1)
def brier(y_true,p):return float(np.mean(np.sum((p-np.eye(3)[y_true])**2,axis=1)))
def ece(y_true,p,bins=15):
    pred=p.argmax(1);conf=p.max(1);correct=pred==y_true;edges=np.linspace(0,1,bins+1);value=0.
    for i in range(bins):
        m=(conf>=edges[i])&(conf<(edges[i+1]) if i<bins-1 else conf<=edges[i+1])
        if m.any():value+=m.mean()*abs(correct[m].mean()-conf[m].mean())
    return float(value)
def higher_quantile(values,alpha):
    level=min(1.,np.ceil((len(values)+1)*(1-alpha))/max(len(values),1));return float(np.quantile(values,level,method="higher"))
def entropy(p):
    p=np.clip(p,1e-8,1);return -np.sum(p*np.log(p),axis=1)/np.log(p.shape[1])

reliability_rows=[];reliability_cases=[]
for key,(bundle,cal_frame) in trained_bundles.items():
    protocol,held_source,method,seed=key
    test_rows=predictions[(predictions.protocol==protocol)&(predictions.held_source==held_source)&
                          (predictions.method==method)&(predictions.seed==seed)].copy()
    if test_rows.empty:continue
    test_ids=test_rows.row_id.to_numpy(int);p_test=test_rows[["p_normal","p_pneumonia","p_tuberculosis"]].to_numpy()
    p_cal=predict_bundle(bundle,X[rows(cal_frame)]);y_cal=y[rows(cal_frame)]
    temperature=fit_temperature(p_cal,y_cal);p_cal=calibrate(p_cal,temperature);p_test=calibrate(p_test,temperature)
    thresholds={k:higher_quantile(1-p_cal[y_cal==k,k],CONFIG["conformal_alpha"]) for k in range(3)}
    sets=[[k for k in range(3) if 1-row[k]<=thresholds[k]] for row in p_test]
    y_test=y[test_ids];covered=np.array([target in s for target,s in zip(y_test,sets)]);sizes=np.array([len(s) for s in sets])
    entropy_threshold=float(np.quantile(entropy(p_cal),CONFIG["selective_entropy_quantile"]))
    pred=p_test.argmax(1);single=np.array([len(s)==1 and s[0]==pred[i] for i,s in enumerate(sets)])
    accepted=single&(entropy(p_test)<=entropy_threshold)
    acc=accuracy_score(y_test[accepted],pred[accepted]) if accepted.any() else np.nan
    row={"protocol":protocol,"held_source":held_source,"method":method,"seed":seed,"temperature":temperature,
         "ece":ece(y_test,p_test),"brier":brier(y_test,p_test),"nll":log_loss(y_test,p_test,labels=np.arange(3)),
         "conformal_coverage":covered.mean(),"mean_set_size":sizes.mean(),"empty_sets":int((sizes==0).sum()),
         "multi_label_sets":int((sizes>1).sum()),"accepted_coverage":accepted.mean(),
         "accepted_errors":int(((pred!=y_test)&accepted).sum()),"selective_risk":1-acc if np.isfinite(acc) else np.nan,
         "referred_n":int((~accepted).sum())}
    for k,name in enumerate(CLASS_NAMES):row[f"{name.lower()}_coverage"]=covered[y_test==k].mean()
    reliability_rows.append(row)
    for i,row_id in enumerate(test_ids):
        reliability_cases.append({"protocol":protocol,"held_source":held_source,"method":method,"seed":seed,"row_id":row_id,
            "true_label":int(y_test[i]),"predicted_label":int(pred[i]),"confidence":float(p_test[i].max()),
            "entropy":float(entropy(p_test[i:i+1])[0]),"conformal_set":"|".join(map(str,sets[i])) if sets[i] else "EMPTY",
            "covered":bool(covered[i]),"accepted":bool(accepted[i]),"source_id":clean_df.loc[row_id,"source_id"],
            **{f"p_{CLASS_NAMES[k].lower()}":float(p_test[i,k]) for k in range(3)}})

reliability_df=pd.DataFrame(reliability_rows);reliability_case_df=pd.DataFrame(reliability_cases)
reliability_df.to_csv(TAB_DIR/"reliability_seed_level.csv",index=False)
reliability_case_df.to_csv(RAW_DIR/"reliability_case_level.csv",index=False)
reliability_summary=(reliability_df.groupby(["protocol","held_source","method"]).mean(numeric_only=True).reset_index())
reliability_summary.to_csv(TAB_DIR/"Table4_Reliability_Summary.csv",index=False)


## 12. Cluster-preserving bootstrap and primary-outcome comparison

The primary contrast is the difference between internal and provenance-held-out macro-F1 for the prespecified primary method. Because the two protocols use different test observations, the contrast is estimated with a stratified bootstrap rather than being mislabeled as paired. Resampling preserves source–class strata and is applied to case-level predictions averaged across seeds. If the held-out experiment was invalid, no confidence interval is fabricated.


In [ ]:
PRIMARY_METHOD="ERM"

def averaged_predictions(frame):
    prob_cols=["p_normal","p_pneumonia","p_tuberculosis"]
    return frame.groupby(["protocol","held_source","row_id","true_label","source_id"])[prob_cols].mean().reset_index()

avg_pred=averaged_predictions(predictions[predictions.method==PRIMARY_METHOD])
bootstrap_rows=[]
if LODO_VALID and (avg_pred.protocol=="provenance_held_out").any():
    internal=avg_pred[avg_pred.protocol=="internal"].copy()
    shifted=avg_pred[avg_pred.protocol=="provenance_held_out"].copy()
    rng=np.random.default_rng(SEED)
    for replicate in range(CONFIG["bootstrap_replicates"]):
        values={}
        for name,frame in (("internal",internal),("provenance_held_out",shifted)):
            sampled=[]
            for _,group in frame.groupby(["source_id","true_label"],dropna=False):
                sampled.append(group.iloc[rng.integers(0,len(group),len(group))])
            boot=pd.concat(sampled,ignore_index=True);p=boot[["p_normal","p_pneumonia","p_tuberculosis"]].to_numpy()
            values[name]=f1_score(boot.true_label,p.argmax(1),average="macro")
        bootstrap_rows.append({"replicate":replicate,**values,
                               "heldout_minus_internal":values["provenance_held_out"]-values["internal"]})
    bootstrap_df=pd.DataFrame(bootstrap_rows);bootstrap_df.to_csv(TAB_DIR/"primary_outcome_bootstrap.csv",index=False)
    primary_ci={"difference_mean":bootstrap_df.heldout_minus_internal.mean(),
                "ci_low":bootstrap_df.heldout_minus_internal.quantile(.025),
                "ci_high":bootstrap_df.heldout_minus_internal.quantile(.975)}
else:
    primary_ci={"status":"not_estimable","reason":guardrail["reason"]}
(RAW_DIR/"primary_outcome.json").write_text(json.dumps(primary_ci,indent=2),encoding="utf-8")
log("Primary outcome: "+json.dumps(primary_ci))

mitigation_table=(experiments.groupby(["protocol","method"]).agg(
    disease_macro_f1=("macro_f1","mean"),worst_source_f1=("worst_source_macro_f1","mean"),
    seed_sd=("macro_f1","std")).reset_index())
primary_source_score=float(intervention_df.loc[intervention_df.intervention==CONFIG["primary_intervention"],"source_macro_f1"].iloc[0])
mitigation_table["input_source_macro_f1"]=primary_source_score
mitigation_table.to_csv(TAB_DIR/"Table3_Mitigation_Comparison.csv",index=False)


## 13. Optional independent external evaluation

External validation is executed only for author-supplied roots. Each root may contain either direct class folders or train/validation/test subfolders. The selected internal model, preprocessing, calibration temperature, conformal thresholds, and referral rule remain locked.


In [ ]:
def scan_external_root(name,root):
    root=Path(root);rows=[]
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in IMAGE_SUFFIXES:continue
        parts=[p.lower() for p in path.parts]
        matches=[folder for folder in FOLDER_TO_CLASS if folder in parts]
        if matches:rows.append({"path":str(path),"class_name":FOLDER_TO_CLASS[matches[-1]],"external_source":name})
    frame=pd.DataFrame(rows)
    if len(frame):frame["label"]=frame.class_name.map(CLASS_TO_INDEX).astype(int)
    return frame

external_results=[]
for external_name,external_root in CONFIG["external_roots"].items():
    external_df=scan_external_root(external_name,external_root)
    if external_df.empty or external_df.class_name.nunique()<2:
        log(f"External source {external_name} skipped: no compatible multi-class files.");continue
    # Extract locked primary-intervention features without changing preprocessing.
    external_df["row_id"]=np.arange(len(external_df))
    encoder=timm.create_model(CONFIG["encoder"],pretrained=True,num_classes=0,global_pool="avg").to(DEVICE).eval()
    for p in encoder.parameters():p.requires_grad=False
    ext_X,_=extract_embeddings(external_df,CONFIG["primary_intervention"]+"_external_"+external_name,encoder)
    encoder.cpu();del encoder
    key=("internal","internal_test",PRIMARY_METHOD,CONFIG["training_seeds"][0])
    if key not in trained_bundles:continue
    bundle,cal_frame=trained_bundles[key];p_ext=predict_bundle(bundle,ext_X)
    p_cal=predict_bundle(bundle,X[rows(cal_frame)]);y_cal=y[rows(cal_frame)];temp=fit_temperature(p_cal,y_cal)
    p_cal=calibrate(p_cal,temp);p_ext=calibrate(p_ext,temp);y_ext=external_df.label.to_numpy()
    thresholds={k:higher_quantile(1-p_cal[y_cal==k,k],CONFIG["conformal_alpha"]) for k in range(3)}
    ext_sets=[[k for k in range(3) if 1-row[k]<=thresholds[k]] for row in p_ext]
    ext_covered=np.array([target in s for target,s in zip(y_ext,ext_sets)]);ext_sizes=np.array([len(s) for s in ext_sets])
    entropy_threshold=float(np.quantile(entropy(p_cal),CONFIG["selective_entropy_quantile"]))
    ext_pred=p_ext.argmax(1);ext_single=np.array([len(s)==1 and s[0]==ext_pred[i] for i,s in enumerate(ext_sets)])
    ext_accepted=ext_single&(entropy(p_ext)<=entropy_threshold)
    ext_acc=accuracy_score(y_ext[ext_accepted],ext_pred[ext_accepted]) if ext_accepted.any() else np.nan
    external_results.append({"external_source":external_name,"n":len(external_df),**basic_metrics(y_ext,p_ext),
        "ece":ece(y_ext,p_ext),"brier":brier(y_ext,p_ext),"conformal_coverage":ext_covered.mean(),
        "mean_set_size":ext_sizes.mean(),"empty_sets":int((ext_sizes==0).sum()),"multi_label_sets":int((ext_sizes>1).sum()),
        "accepted_coverage":ext_accepted.mean(),"accepted_errors":int(((ext_pred!=y_ext)&ext_accepted).sum()),
        "selective_risk":1-ext_acc if np.isfinite(ext_acc) else np.nan,"referred_n":int((~ext_accepted).sum())})

pd.DataFrame(external_results).to_csv(TAB_DIR/"external_validation.csv",index=False)
if not external_results:log("External validation not run: no independent external root was supplied or eligible.")


## 14. Manuscript-ready PNG figures

The notebook creates up to six figures aligned with the manuscript. Figure 6 is produced only if external validation exists. No placeholder numbers are drawn into figures.


In [ ]:
sns.set_theme(style="whitegrid",context="notebook")

# Figure 1: provenance reconstruction and class support.
fig,axes=plt.subplots(1,2,figsize=(13,4.8))
assignment_counts=clean_df.assignment_type.value_counts().rename_axis("Assignment").rename("Images").reset_index()
sns.barplot(data=assignment_counts,x="Assignment",y="Images",ax=axes[0],color="#2E6F95");axes[0].tick_params(axis="x",rotation=20);axes[0].set_title("Provenance assignment")
sns.heatmap(source_class[CLASS_NAMES],annot=True,fmt="d",cmap="Blues",ax=axes[1]);axes[1].set_title("Diagnostic class support by source/domain");axes[1].set_xlabel("Class");axes[1].set_ylabel("Source/domain")
fig.tight_layout();fig.savefig(FIG_DIR/"Figure1_Provenance_Reconstruction.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

# Figure 2: source-label dependence and source predictability.
fig,axes=plt.subplots(1,2,figsize=(13,4.8))
row_norm=source_class[CLASS_NAMES].div(source_class[CLASS_NAMES].sum(axis=1),axis=0)
sns.heatmap(row_norm,annot=True,fmt=".2f",cmap="mako",vmin=0,vmax=1,ax=axes[0]);axes[0].set_title("Within-source class composition")
sns.barplot(data=source_predictability_df,x="representation",y="source_macro_f1",ax=axes[1],color="#2A9D8F");axes[1].tick_params(axis="x",rotation=35);axes[1].set_ylim(0,1);axes[1].set_title("Source predictability")
fig.tight_layout();fig.savefig(FIG_DIR/"Figure2_Source_Class_Dependence.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

# Figure 3: internal versus held-out discrimination.
fig,ax=plt.subplots(figsize=(9,5))
plot_disc=(experiments.groupby(["protocol","method"]).macro_f1.agg(["mean","std"]).reset_index())
for i,protocol in enumerate(plot_disc.protocol.unique()):
    d=plot_disc[plot_disc.protocol==protocol];x=np.arange(len(d))+(i-.5)*.18
    ax.errorbar(x,d["mean"],yerr=d["std"].fillna(0),fmt="o",capsize=4,label=protocol)
ax.set_xticks(np.arange(len(d)));ax.set_xticklabels(d.method,rotation=20);ax.set_ylim(0,1);ax.set_ylabel("Macro-F1");ax.set_title("Internal versus provenance-held-out discrimination");ax.legend()
fig.tight_layout();fig.savefig(FIG_DIR/"Figure3_Internal_vs_HeldOut.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

# Figure 4: intervention trade-off.
fig,ax=plt.subplots(figsize=(7,5.5));sns.scatterplot(data=intervention_df,x="source_macro_f1",y="macro_f1",hue="intervention",s=130,ax=ax)
for _,r in intervention_df.iterrows():ax.annotate(r.intervention,(r.source_macro_f1,r.macro_f1),xytext=(4,4),textcoords="offset points",fontsize=8)
ax.set(xlabel="Source-prediction macro-F1 (lower is preferable)",ylabel="Disease macro-F1 (higher is preferable)",title="Source-neutralization trade-off");ax.legend_.remove()
fig.tight_layout();fig.savefig(FIG_DIR/"Figure4_Source_Interventions.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

# Figure 5: reliability under shift using averaged seed results.
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
rel_plot=reliability_summary.groupby(["protocol","method"]).mean(numeric_only=True).reset_index()
sns.barplot(data=rel_plot,x="method",y="ece",hue="protocol",ax=axes[0]);axes[0].tick_params(axis="x",rotation=25);axes[0].set_title("Calibration error")
sns.barplot(data=rel_plot,x="method",y="conformal_coverage",hue="protocol",ax=axes[1]);axes[1].axhline(1-CONFIG["conformal_alpha"],color="black",ls="--");axes[1].tick_params(axis="x",rotation=25);axes[1].set_title("Conformal coverage")
sns.barplot(data=rel_plot,x="method",y="selective_risk",hue="protocol",ax=axes[2]);axes[2].tick_params(axis="x",rotation=25);axes[2].set_title("Selective risk")
for ax in axes:
    if ax.legend_:ax.legend_.remove()
handles,labels=axes[0].get_legend_handles_labels();fig.legend(handles,labels,loc="upper center",ncol=max(1,len(labels)),bbox_to_anchor=(.5,1.03))
fig.tight_layout();fig.savefig(FIG_DIR/"Figure5_Reliability_Under_Shift.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

if external_results:
    ext=pd.DataFrame(external_results);fig,ax=plt.subplots(figsize=(8,4.8));
    ext.melt(id_vars=["external_source"],value_vars=["macro_f1","balanced_accuracy","ece","brier"],var_name="Metric",value_name="Value").pipe(
        lambda d:sns.barplot(data=d,x="external_source",y="Value",hue="Metric",ax=ax))
    ax.set_title("Locked external validation");fig.tight_layout();fig.savefig(FIG_DIR/"Figure6_External_Validation.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)


## 15. Radiologist-review manifest, result archive, and manuscript summary

A review sheet is generated from source-sensitive errors and referrals. Clinical ratings remain blank for human completion. The final archive excludes source images and contains only manifests, predictions, tables, figures, models, and configuration files.


In [ ]:
if not reliability_case_df.empty:
    review=reliability_case_df[(reliability_case_df.method==PRIMARY_METHOD)&(reliability_case_df.seed==CONFIG["training_seeds"][0])].copy()
    review["error"]=review.predicted_label!=review.true_label
    review["review_priority"]=np.select([review.error&review.accepted,review.error,~review.accepted],
                                        ["high_confidence_error","error","referral"],default="stable_correct")
    review=(review.groupby("review_priority",group_keys=False).apply(lambda d:d.sample(min(25,len(d)),random_state=SEED))
            .reset_index(drop=True))
    review=review.merge(clean_df[["row_id","path","class_name","assignment_type","assignment_confidence"]],on="row_id",how="left")
    for col in ["radiologist_anatomical_plausibility","artifact_influence","image_quality","review_comment"]:review[col]=""
    review.to_csv(TAB_DIR/"radiologist_review_manifest.csv",index=False)

summary_lines=[
    "THORACIC RADIOLOGY SOURCE-AWARE CXR EXPERIMENTS — RUN COMPLETE",
    f"Timestamp: {STAMP}",f"Dataset root: {DATA_ROOT}",f"Output root: {OUTPUT_DIR}",
    f"Retained images: {len(clean_df):,}",f"Sources/domains: {clean_df.source_id.nunique()}",
    f"Source-label Cramer's V: {association['cramers_v']:.6f}",
    f"Source-label NMI: {association['normalized_mutual_information']:.6f}",
    f"Eligible three-class held-out sources: {eligible_sources}",
    f"Three-class held-out evaluation valid: {LODO_VALID}",
    f"Primary outcome: {primary_ci}","",
    "INTERVENTION RESULTS",intervention_df.round(6).to_string(index=False),"",
    "DISCRIMINATION SUMMARY",discrimination_summary.round(6).to_string(index=False),"",
    "RELIABILITY SUMMARY",reliability_summary.round(6).to_string(index=False),"",
    "GUARDRAILS",
    "- Latent domains are technical clusters, not verified institutions.",
    "- No three-class source-held-out result is reported when class support is inadequate.",
    "- External validation is absent unless an independent external root was explicitly supplied.",
    "- Calibration and conformal coverage under source shift are empirical diagnostics, not unconditional guarantees.",
    "- Retrospective public-data performance does not establish clinical utility.",
]
LOG_PATH.write_text("\n".join(summary_lines),encoding="utf-8")
print("\n".join(summary_lines))

archive_path=OUTPUT_DIR.parent/f"{STAMP}_Thoracic_Radiology_Source_Aware_CXR_results.zip"
with zipfile.ZipFile(archive_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():archive.write(path,path.relative_to(OUTPUT_DIR))
print("Results archive saved to Google Drive:",archive_path)

if CONFIG["auto_download_results"] and IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))


## Manuscript-use checklist

Before transferring values into the article:

- Verify `source_heldout_guardrail.json` and do not report an invalid three-class held-out result.
- Use `Table1_Provenance_Characteristics.csv`, `Table2_Discrimination_Summary.csv`, `Table3_Mitigation_Comparison.csv`, and `Table4_Reliability_Summary.csv` as the four main tables.
- Report point estimates with the primary bootstrap interval, not only seed averages.
- Keep deterministic and latent provenance separate in the text.
- Interpret a reduction in source predictability jointly with disease and worst-source performance.
- If `external_validation.csv` is empty, state explicitly that no eligible external cohort was evaluated.
- Do not describe the radiologist-review manifest as reviewed until its blank clinical fields have been completed.
- Preserve the distinction between technical robustness and clinical generalization.
